In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t

In [ ]:
df = pd.read_csv('<PATH_TO_FOR_GLM_DRUG_RESPONSE_TSV>', sep = '\t')

In [ ]:
def run_additive_models(df, snps, phenotypes):
    results = []
    for snp in snps:
        for pheno in phenotypes:
            sub = df[[snp, pheno]].dropna()
            X = sm.add_constant(sub[snp].astype(float).values)
            y = sub[pheno].astype(float).values
            model = sm.OLS(y, X, missing='drop').fit()
            results.append({
                \"snp\": snp,
                \"phenotype\": pheno,
                \"beta\": model.params[1],
                \"pvalue\": model.pvalues[1],
                \"n\": len(sub),
            })
    return pd.DataFrame(results)

In [ ]:
snp_list = ["rs12513069","SLC35F2","PKD1"]
pheno_list = [
   'num_arb_therapies_ADJ','num_diff_drug_classes_ADJ','num_diff_arbs_ADJ','longest_arb_duration_ADJ',
   'avg_dose_Candesartan Cilexetil_ADJ','avg_dose_Eprosartan_ADJ','avg_dose_Irbesartan_ADJ',
   'avg_dose_Losartan Potassium_ADJ','avg_dose_Olmesartan_ADJ','avg_dose_Telmisartan_ADJ',
   'avg_dose_Valsartan_ADJ','dose_ever_increased_ADJ','arb_is_longest_therapy_ADJ','arb_is_last_therapy_ADJ',
   'arb_is_first_therapy_ADJ','arb_ever_augmented_ADJ','changed_from_arb_ADJ'
 ]

results = run_additive_models(df, snp_list, pheno_list)

In [ ]:
def plot_three_bars(
    df,
    genotype_col=\"rs12513069\",
    columns_to_plot=(
        \"arb_is_first_therapy_ADJ\",
        \"dose_ever_increased_ADJ\",
        \"num_diff_drug_classes_ADJ\"
    ),
    col_labels=(
        \"ARB was first therapy\",
        \"ARB dose ever increased\",
        \"Number of drug classes prescribed\"
    ),
    pvals=(0.075621, 0.176313, 0.013481),
    output_png=\"<PATH_TO_THREE_BARPLOTS_95CI_WHITE_PNG>\"
):
    df_plot = df.copy()
    df_plot[\"genotype_numeric\"] = pd.to_numeric(df_plot[genotype_col], errors=\"coerce\")

    bar_colors = [\"#728A95\", \"#AEB8AF\", \"#D2D9D5\"]
    numeric_genotypes = [0.0, 1.0, 2.0]
    genotype_labels = [\"0/0\", \"0/1\", \"1/1\"]

    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12,4), facecolor=\"white\")
    axes = axes.flatten()

    for i, (ax, col_name, col_label, pval) in enumerate(zip(axes, columns_to_plot, col_labels, pvals)):
        sub = df_plot[[col_name, \"genotype_numeric\"]].dropna()
        sub = sub[sub[\"genotype_numeric\"].isin(numeric_genotypes)]

        means = []
        ci_95s = []
        for gval in numeric_genotypes:
            rows = sub.loc[sub[\"genotype_numeric\"] == gval, col_name].astype(float)
            mu = rows.mean()
            sd = rows.std()
            n = len(rows)
            se = sd / np.sqrt(n)
            crit = t.ppf(0.975, n - 1)
            ci_95 = crit * se
            means.append(mu)
            ci_95s.append(ci_95)

        for j, (m, c) in enumerate(zip(means, ci_95s)):
            ax.bar(j, m, yerr=c, capsize=4, color=bar_colors[j])

        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(genotype_labels)
        ax.set_ylabel(\"z-score\")
        if i == 1:
            ax.set_xlabel(\"rs12513069 genotype\")
        ax.set_title(col_label)
        ax.axhline(y=0, color=\"black\", linewidth=1)

        if i == 0:
            x_anno, ha_pos = 0.05, \"left\"
        elif i == 1:
            x_anno, ha_pos = 0.5, \"center\"
        else = 0.95, \"right\"

        ax.text(
            x_anno,
            0.95,
            f\"p={pval:.3g}\",
            ha=ha_pos,
            va=\"top\",
            fontsize=11,
            transform=ax.transAxes
        )

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(\"black\")

    for ax in axes:
        ax.set_facecolor(\"white\")

    plt.tight_layout()
    plt.savefig(output_png, dpi=150, bbox_inches=\"tight\")
    plt.show()

In [ ]:
plot_three_bars(
    df=df,
    genotype_col="rs12513069",
    columns_to_plot=(
        "arb_is_first_therapy_ADJ",
        "dose_ever_increased_ADJ",
        "num_diff_drug_classes_ADJ"
    ),
    pvals=(0.0756, 0.176, 0.0135),
    output_png="<PATH_TO_THREE_BARPLOTS_PNG>"
)